# Colab 실행 안내

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_01_pytorch_core/01_autograd_essentials.ipynb)

이 노트북은 **PyTorch autograd 가 어떻게 동작하는지**를 코드로 시연한다.
이미 numpy 로 backward 를 손으로 구현해 본 사람이 PyTorch 의 `loss.backward()` 한 줄을
"마법" 이 아니라 **chain rule 의 자동화** 로 이해하는 것이 목표다.

흐름:

1. `requires_grad` 와 leaf tensor
2. 계산 그래프 (computational graph) 와 `.grad_fn`
3. `.grad` 의 누적 동작
4. `detach()` 와 `with torch.no_grad()`
5. 수치 미분으로 autograd 검증
6. 고차 미분 (`create_graph=True`)
7. 벡터 출력의 backward (vector-Jacobian product)
8. 흔한 함정 (Common Pitfalls)

> 책의 numpy 구현에서 손으로 적었던 `_input_grad`, `_param_grad` 가 PyTorch 에서는
> "각 텐서가 자신을 만든 함수(`grad_fn`) 를 기억" 하는 방식으로 자동화되어 있다.
> 이 노트북은 그 내부를 들여다본다.

In [ ]:
# 공통 실행 환경 준비 (Colab/Local 통합)
# 이 셀은 Colab과 로컬 Jupyter 사이의 환경 차이를 흡수한다.
# DLFS_code 의 prepare_notebook_environment 는 lincoln 폴더에 의존하므로
# study_* 노트북에서는 더 단순한 부트스트랩을 직접 둔다.
from pathlib import Path
import subprocess
import sys
import os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_01_pytorch_core'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

# Colab 에는 PyTorch 가 기본 설치되어 있지만 로컬에 없을 수도 있어 안전하게 보강.
try:
    import torch  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa: F401

print(f'torch version: {torch.__version__}')

In [ ]:
# 핵심 import. 재현성 확보를 위해 seed 고정.
import numpy as np
import torch

SEED = 20260427
np.random.seed(SEED)
torch.manual_seed(SEED)

# 1. `requires_grad` 와 leaf tensor

PyTorch 의 autograd 는 **leaf tensor** 와 **non-leaf tensor** 를 다르게 다룬다.

- **leaf tensor**: 사용자가 직접 만든 텐서 (`torch.tensor(...)`, `torch.zeros(...)`).
  `requires_grad=True` 로 만들면 backward 후 `.grad` 에 $\partial L / \partial x$ 가 저장된다.
- **non-leaf tensor**: 다른 텐서로부터 연산되어 생긴 텐서 (예: `y = x ** 2`).
  기본적으로 `.grad` 는 None — 메모리 절약을 위해서다.
  값이 필요하면 `y.retain_grad()` 를 backward 전에 호출해야 한다.

수식으로:

$$L = f(x), \quad x \text{ 는 leaf} \;\Rightarrow\; \text{backward 후 } x.\text{grad} = \frac{\partial L}{\partial x}$$

In [ ]:
# leaf vs non-leaf 데모
x = torch.tensor(2.0, requires_grad=True)   # leaf
y = x ** 2                                  # non-leaf (x 로부터 계산됨)
z = y + 3                                   # non-leaf
L = z * 1.0                                 # 스칼라 손실

print('x.is_leaf =', x.is_leaf, ', y.is_leaf =', y.is_leaf)

L.backward()
print('x.grad =', x.grad)   # dL/dx = 2x = 4
print('y.grad =', y.grad)   # None — non-leaf 는 기본적으로 안 저장

# 비교: y.grad 도 받고 싶다면 backward 전에 retain_grad() 호출
x2 = torch.tensor(2.0, requires_grad=True)
y2 = x2 ** 2
y2.retain_grad()
L2 = y2 + 3
L2.backward()
print('y2.grad =', y2.grad, '(retain_grad 호출 후)')

# 2. 계산 그래프 (computational graph) 와 `.grad_fn`

PyTorch 는 **define-by-run** 방식이다. forward 가 실행되는 순간 그래프가 동적으로 만들어진다.
각 non-leaf 텐서는 자기를 만든 연산을 `grad_fn` 으로 기억한다.

$$f \circ g \circ h \text{ 의 backward} \;=\; f' \cdot g' \cdot h'$$

이 chain rule 적용을 PyTorch 가 `grad_fn` 그래프를 거꾸로 따라가며 자동으로 한다.
numpy 구현에서 우리가 `for op in reversed(self.operations): op.backward(...)` 라고 적었던 그 부분이다.

In [ ]:
# .grad_fn 출력 + numpy 손계산과 비교
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2          # MulBackward / PowBackward
z = torch.sin(y)    # SinBackward
L = z + x           # AddBackward

print('y.grad_fn =', y.grad_fn)
print('z.grad_fn =', z.grad_fn)
print('L.grad_fn =', L.grad_fn)

L.backward()
torch_grad = x.grad.item()

# numpy 로 직접 계산: L = sin(x^2) + x  =>  dL/dx = cos(x^2) * 2x + 1
xv = 3.0
numpy_grad = np.cos(xv ** 2) * 2 * xv + 1
print(f'torch dL/dx = {torch_grad:.6f}')
print(f'numpy dL/dx = {numpy_grad:.6f}')
print('일치 여부:', np.isclose(torch_grad, numpy_grad))

# 3. `.grad` 는 누적된다

PyTorch 의 `.grad` 는 backward 호출시마다 **덮어쓰지 않고 누적(`+=`)** 한다.
이는 같은 파라미터에 대해 여러 손실의 그래디언트를 합산하거나, gradient accumulation 으로
큰 배치를 시뮬레이션할 때 유용하다.

하지만 일반적인 학습 루프에서는 매 step `optimizer.zero_grad()` 또는 `param.grad = None` 으로
초기화하지 않으면 그래디언트가 무한정 쌓여 학습이 망가진다.

In [ ]:
# zero_grad 없이 두 번 backward — grad 가 두 배로 누적된다
x = torch.tensor(2.0, requires_grad=True)

L1 = x ** 2
L1.backward()
print('1차 backward 후 x.grad =', x.grad.item())   # dL1/dx = 2x = 4

L2 = x ** 2
L2.backward()
print('2차 backward 후 x.grad =', x.grad.item())   # 4 + 4 = 8 (누적!)

# 초기화 후 다시 — 의도한 값이 나옴
x.grad = None
L3 = x ** 2
L3.backward()
print('grad 초기화 후 backward x.grad =', x.grad.item())   # 4

# 4. `detach()` 와 `with torch.no_grad()`

둘 다 그래프 추적을 끊지만 **단위가 다르다**.

- `tensor.detach()`: **텐서 한 개**를 그래프에서 떼어낸 새 텐서를 반환.
  데이터는 공유하되 `requires_grad=False`. 모델 일부 출력만 얼리고 싶을 때.
- `with torch.no_grad():`: **컨텍스트 안의 모든 연산** 에 대해 그래프 기록을 끔.
  추론 루프에서 메모리/속도 절약 목적.

추론 시에는 `with torch.no_grad():` 가 거의 항상 정답이다.

In [ ]:
# detach vs no_grad 메모리/그래프 차이 시연
import time

x = torch.randn(1000, 1000, requires_grad=True)

# (a) 그냥 forward — 그래프 만들어짐
t0 = time.perf_counter()
y = (x * x).sum()
t1 = time.perf_counter()
print(f'그래프 추적 O   : {(t1 - t0)*1000:.2f} ms, y.requires_grad = {y.requires_grad}')

# (b) detach — x 한 개만 그래프에서 분리
t0 = time.perf_counter()
y_det = (x.detach() * x.detach()).sum()
t1 = time.perf_counter()
print(f'detach 사용     : {(t1 - t0)*1000:.2f} ms, y_det.requires_grad = {y_det.requires_grad}')

# (c) no_grad — 컨텍스트 안 모든 연산이 그래프 추적 OFF
t0 = time.perf_counter()
with torch.no_grad():
    y_ng = (x * x).sum()
t1 = time.perf_counter()
print(f'no_grad 사용    : {(t1 - t0)*1000:.2f} ms, y_ng.requires_grad = {y_ng.requires_grad}')

# 5. 수치 미분으로 autograd 검증

이론적으로 자동미분이 옳더라도, 직접 한번 확인해 보면 안심이 된다.
**중심차분(central difference)** 으로 수치 미분과 autograd 결과를 비교한다.

$$f'(x) \;\approx\; \frac{f(x+\epsilon) - f(x-\epsilon)}{2\epsilon}$$

PyTorch 가 제공하는 `torch.autograd.gradcheck` 는 이 비교를 자동화한 도구다.

In [ ]:
# (a) 직접 finite difference 로 검증
def f(x):
    # f(x) = sin(x) * x^2
    return torch.sin(x) * x ** 2

x = torch.tensor(1.3, dtype=torch.float64, requires_grad=True)   # gradcheck 는 float64 권장
L = f(x)
L.backward()
autograd_grad = x.grad.item()

eps = 1e-6
with torch.no_grad():
    fd_grad = (f(x + eps) - f(x - eps)).item() / (2 * eps)

print(f'autograd grad     = {autograd_grad:.10f}')
print(f'finite difference = {fd_grad:.10f}')
print(f'차이 = {abs(autograd_grad - fd_grad):.2e}')

# (b) gradcheck 사용 — 텐서 입력 함수의 모든 성분에 대해 자동 비교
g = lambda t: (torch.sin(t) * t ** 2).sum()
t_in = torch.randn(5, dtype=torch.float64, requires_grad=True)
ok = torch.autograd.gradcheck(g, (t_in,), eps=1e-6, atol=1e-5)
print('gradcheck pass:', ok)

# 6. 고차 미분 (`create_graph=True`)

backward 자체도 미분 가능한 연산으로 기록하면 **고차 미분** 을 얻을 수 있다.
Hessian, 메타러닝(MAML), implicit differentiation 의 기초다.

$$f(x) = x^3 \;\Rightarrow\; f'(x) = 3x^2, \;\; f''(x) = 6x$$

In [ ]:
# 1차 미분이 다시 미분 가능하도록 create_graph=True
x = torch.tensor(2.0, requires_grad=True)
y = x ** 3

# 1차 미분: dy/dx = 3x^2 = 12  (create_graph=True 로 그래프 보존)
grad1 = torch.autograd.grad(y, x, create_graph=True)[0]
print('1차 미분 dy/dx =', grad1.item())

# 2차 미분: d2y/dx2 = 6x = 12
grad2 = torch.autograd.grad(grad1, x)[0]
print('2차 미분 d2y/dx2 =', grad2.item())

# 7. 벡터 출력의 backward — vector-Jacobian product

`y.backward()` 는 `y` 가 **스칼라** 일 때만 인자 없이 동작한다.
`y` 가 벡터/텐서이면 "어느 방향으로 미분할지" 를 `grad_outputs` 로 알려줘야 한다.

기하학적으로 PyTorch 가 계산하는 것은 **vector-Jacobian product (VJP)**:

$$v^\top J \quad \text{where} \quad J = \frac{\partial y}{\partial x}, \;\; v = \text{grad\_outputs}$$

즉 우리는 보통 `loss.sum().backward()` 처럼 스칼라로 줄이거나, `grad_outputs=torch.ones_like(y)` 를 줘서 같은 효과를 낸다.

In [ ]:
# 벡터 y 에 대해 grad_outputs 주는 예
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = x ** 2                              # y = [1, 4, 9], 벡터

# y.backward()  # <- 스칼라가 아니라 에러 발생
v = torch.tensor([1.0, 1.0, 1.0])       # 모든 출력에 동일 가중치 → sum 과 동일
y.backward(gradient=v)
print('grad_outputs=ones 일 때 x.grad =', x.grad)   # dy_i/dx_i = 2x_i → [2, 4, 6]

# 다른 가중치
x.grad = None
y2 = x ** 2
v2 = torch.tensor([1.0, 0.0, 0.0])      # y_0 만 신경 씀
y2.backward(gradient=v2)
print('grad_outputs=[1,0,0] 일 때 x.grad =', x.grad)   # [2, 0, 0]

# 8. 흔한 함정 (Common Pitfalls)

실제 학습 코드에서 자주 부딪히는 함정들:

1. **leaf 에 in-place 연산**: `x += 1` 같은 in-place 가 backward 그래프를 깨뜨려 에러.
2. **정수 dtype + requires_grad**: 정수 텐서는 미분 불가. dtype 을 float 로.
3. **`optimizer.zero_grad()` 누락**: grad 누적으로 학습 폭주.
4. **loss 텐서 그대로 list 에 누적**: 그래프가 살아 있어 메모리 폭발. 반드시 `loss.item()` 으로 스칼라 추출.

각각 짧게 시연한다.

In [ ]:
# 함정 1: leaf 에 in-place 연산
try:
    x = torch.tensor(2.0, requires_grad=True)
    x += 1                          # leaf 에 직접 in-place — 위험
    L = x ** 2
    L.backward()
except RuntimeError as e:
    print('[함정 1] leaf in-place 에러:', str(e).split('.')[0])

# 안전한 방법: 새 텐서로 받기
x = torch.tensor(2.0, requires_grad=True)
x = x + 1                           # x 는 이제 non-leaf 이지만 backward 는 정상
L = x ** 2
L.backward()
print('[함정 1] 우회 OK, grad 흐름 정상')

In [ ]:
# 함정 2: 정수 dtype 으로는 requires_grad 자체가 불가
try:
    x_int = torch.tensor([1, 2, 3], requires_grad=True)
except RuntimeError as e:
    print('[함정 2] 정수 텐서 + requires_grad 에러:', str(e).split('.')[0])

# 해결: dtype 을 float 으로
arr = np.array([1, 2, 3])
x_ok = torch.tensor(arr, dtype=torch.float32, requires_grad=True)
print('[함정 2] float dtype 으로 OK, requires_grad =', x_ok.requires_grad)

In [ ]:
# 함정 3: zero_grad 누락 시뮬레이션
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(SEED)
model = nn.Linear(2, 1)
opt = optim.SGD(model.parameters(), lr=0.0)   # lr=0 으로 step 영향 제거, grad 누적만 본다
xb = torch.randn(4, 2)
yb = torch.randn(4, 1)
loss_fn = nn.MSELoss()

for step in range(3):
    # opt.zero_grad()   # <-- 의도적으로 빠뜨림
    pred = model(xb)
    loss = loss_fn(pred, yb)
    loss.backward()
    g = model.weight.grad.norm().item()
    print(f'[함정 3] step {step}: weight.grad norm = {g:.4f}  (zero_grad 빠지면 누적)')

In [ ]:
# 함정 4: loss 텐서를 그래프째로 list 에 쌓기 vs .item() 으로 스칼라만 쌓기
import gc

torch.manual_seed(SEED)
model = nn.Linear(100, 100)
opt = optim.SGD(model.parameters(), lr=1e-3)

bad_history = []   # 텐서를 그대로 누적 — 그래프가 살아 있음
ok_history  = []   # .item() 으로 float 만 누적

xb = torch.randn(32, 100)
yb = torch.randn(32, 100)
loss_fn = nn.MSELoss()

for _ in range(5):
    opt.zero_grad()
    loss = loss_fn(model(xb), yb)
    loss.backward()
    opt.step()
    bad_history.append(loss)         # 위험: loss.grad_fn 이 아직 살아 있다
    ok_history.append(loss.item())   # 안전: float 스칼라만 보관

print('[함정 4] bad_history[0].grad_fn =', bad_history[0].grad_fn)
print('[함정 4] ok_history[0] =', ok_history[0], '(단순 float)')
print('         학습 루프가 길어지면 bad_history 가 그래프를 잡고 있어 OOM 발생.')

# 정리

| 개념 | 핵심 |
|------|------|
| `requires_grad` | leaf 텐서에 켜야 `.grad` 가 채워진다 |
| `grad_fn` | 텐서가 자기를 만든 연산을 기억 → 역방향 그래프 |
| `.grad` 누적 | 매 step `zero_grad()` 또는 `param.grad = None` 필요 |
| `detach()` / `no_grad()` | 텐서 단위 / 컨텍스트 단위로 그래프 추적 OFF |
| `gradcheck` | 수치 미분으로 autograd 결과를 자동 검증 |
| `create_graph=True` | 1차 미분을 다시 미분 가능하게 해 고차 미분 가능 |
| `grad_outputs` | 벡터 출력의 backward 는 VJP, 가중치 벡터를 줘야 함 |

다음 노트북(`02_training_pipeline.ipynb`) 에서는 이 autograd 위에 얹어진
`nn.Module`, `Optimizer`, `DataLoader` 의 부품들을 분해해 본다.